In [ ]:
# ── CELL 1: GPU + deps ────────────────────────────────────────────────────────
!nvidia-smi
!pip install pytorch-metric-learning -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/results/checkpoints', exist_ok=True)
print('Ready.')

In [ ]:
# ── CELL 3: Dataset ───────────────────────────────────────────────────────────
import os, glob
candidates = glob.glob('/content/data/**/PartAnnotation', recursive=True)
if candidates:
    DATA_ROOT = candidates[0]
    print('Dataset already present:', DATA_ROOT)
else:
    KAGGLE_USERNAME = "PASTE_YOUR_USERNAME"
    KAGGLE_KEY      = "PASTE_YOUR_KEY"
    import json
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY']      = KAGGLE_KEY
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    os.makedirs('/content/data', exist_ok=True)
    os.chdir('/content/data')
    !pip install kaggle -q
    !kaggle datasets download -d majdouline20/shapenetpart-dataset
    !unzip -q shapenetpart-dataset.zip
    os.chdir('/content')
    candidates = glob.glob('/content/data/**/PartAnnotation', recursive=True)
    DATA_ROOT = candidates[0]
print('DATA_ROOT:', DATA_ROOT)

In [ ]:
# ── CELL 4: Dataset loader ────────────────────────────────────────────────────
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

SYNSET_TO_CLASS = {
    '02691156':0,'02773838':1,'02954340':2,'02958343':3,'03001627':4,
    '03261776':5,'03467517':6,'03624134':7,'03636649':8,'03642806':9,
    '03790512':10,'03797390':11,'03948459':12,'04099429':13,'04225987':14,'04379243':15
}

def load_pts(path):
    pts = []
    with open(path) as f:
        for line in f:
            v = line.strip().split()
            if len(v) >= 3: pts.append([float(x) for x in v[:3]])
    return np.array(pts, dtype='float32')

def load_seg(path):
    segs = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s: segs.append(int(s))
    return np.array(segs, dtype='int64')

class ShapeNet_coseg(Dataset):
    def __init__(self, partition='train', num_points=1024, obj_class=4,
                 data_root=None, train_ratio=0.8):
        self.num_points = num_points
        if data_root is None: data_root = DATA_ROOT
        target_syn = next((s for s,c in SYNSET_TO_CLASS.items() if c==obj_class), None)
        syn_dir = os.path.join(data_root, target_syn)
        pts_dir = os.path.join(syn_dir, 'points')
        seg_dir = os.path.join(syn_dir, 'points_label')
        all_ids = sorted([f[:-4] for f in os.listdir(pts_dir) if f.endswith('.pts')])
        seg_map = {}
        for dirpath, _, files in os.walk(seg_dir):
            for f in files:
                if f.endswith('.seg') and f[:-4] not in seg_map:
                    seg_map[f[:-4]] = os.path.join(dirpath, f)
        valid_ids = [i for i in all_ids if i in seg_map]
        np.random.seed(42)
        perm  = np.random.permutation(len(valid_ids))
        split = int(len(valid_ids) * train_ratio)
        chosen = [valid_ids[i] for i in (perm[:split] if partition=='train' else perm[split:])]
        self.samples = [(os.path.join(pts_dir,sid+'.pts'), seg_map[sid]) for sid in chosen]
        print(f'[ShapeNet] {partition}: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        pc  = load_pts(self.samples[idx][0])
        seg = load_seg(self.samples[idx][1])
        n   = min(len(pc), len(seg))
        pc, seg = pc[:n], seg[:n]
        N   = len(pc)
        idx_s = np.random.choice(N, self.num_points, replace=(N < self.num_points))
        pc, seg = pc[idx_s], seg[idx_s]
        pc -= pc.mean(0); s = np.max(np.linalg.norm(pc, axis=1))
        if s > 0: pc /= s
        binary = (seg > seg.min()).astype('int64')
        return pc.astype('float32'), binary

ds_tmp = ShapeNet_coseg('test', 1024, 4)
pc0, lb0 = ds_tmp[0]
print(f'Sample: {pc0.shape}, fg%={lb0.mean()*100:.1f}%')

In [ ]:
# ── CELL 5: Model ─────────────────────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F
import copy

def knn_graph(x, k):
    xt = x.permute(0,2,1)
    return torch.cdist(xt,xt).topk(k+1,dim=-1,largest=False).indices[:,:,1:]

def get_edge_feature(x, idx):
    B,D,N = x.shape; k = idx.shape[2]
    xt   = x.permute(0,2,1).contiguous()
    flat = idx.reshape(B,-1)
    nb   = torch.gather(xt,1,flat.unsqueeze(-1).expand(B,N*k,D)).view(B,N,k,D)
    xi   = xt.unsqueeze(2).expand(B,N,k,D)
    return torch.cat([xi, nb-xi], dim=-1).permute(0,3,1,2)

class EdgeConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=20):
        super().__init__()
        self.k   = k
        self.net = nn.Sequential(
            nn.Conv2d(in_ch*2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2))
    def forward(self, x):
        return self.net(get_edge_feature(x, knn_graph(x, self.k))).max(dim=-1)[0]

class DGCNN(nn.Module):
    def __init__(self, k=20, emb_dim=512):
        super().__init__()
        self.ec1 = EdgeConv(3,   64,  k)
        self.ec2 = EdgeConv(64,  64,  k)
        self.ec3 = EdgeConv(64,  128, k)
        self.ec4 = EdgeConv(128, 256, k)
        self.proj = nn.Sequential(
            nn.Conv1d(512, emb_dim, 1, bias=False),
            nn.BatchNorm1d(emb_dim), nn.LeakyReLU(0.2), nn.Dropout(0.4))
    def forward(self, x):
        f1=self.ec1(x); f2=self.ec2(f1); f3=self.ec3(f2); f4=self.ec4(f3)
        return self.proj(torch.cat([f1,f2,f3,f4],dim=1))

class PointSampler(nn.Module):
    def __init__(self, in_dim, n_sample):
        super().__init__()
        self.n   = n_sample
        self.net = nn.Sequential(
            nn.Linear(in_dim,256), nn.ReLU(),
            nn.Linear(256,128),    nn.ReLU(),
            nn.Linear(128,1))
    def forward(self, feats):
        B,D,N = feats.shape
        w   = torch.softmax(self.net(feats.permute(0,2,1)).squeeze(-1), dim=-1)
        idx = w.topk(self.n, dim=-1).indices
        return torch.gather(feats,2,idx.unsqueeze(1).expand(B,D,self.n)), idx, w

class PartHead(nn.Module):
    def __init__(self, in_dim, n_parts=2):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_parts)
    def forward(self, feats):
        return torch.softmax(self.fc(feats.permute(0,2,1)), dim=-1)

class CoSegNet(nn.Module):
    def __init__(self, n_fg=256, n_bg=256, k=20, emb_dim=512, n_parts=2):
        super().__init__()
        self.encoder    = DGCNN(k=k, emb_dim=emb_dim)
        self.fg_sampler = PointSampler(emb_dim, n_fg)
        self.bg_sampler = PointSampler(emb_dim, n_bg)
        self.part_head  = PartHead(emb_dim, n_parts)
    def forward(self, xyz):
        feats = self.encoder(xyz.permute(0,2,1))
        fg_f, fg_idx, fg_w = self.fg_sampler(feats)
        bg_f, bg_idx, bg_w = self.bg_sampler(feats)
        probs = self.part_head(feats)
        return feats, fg_f, bg_f, fg_w, bg_w, probs

print('Model defined.')

In [ ]:
# ── CELL 6: Loss functions ─────────────────────────────────────────────────────
from pytorch_metric_learning.losses import NTXentLoss
ntxent = NTXentLoss(temperature=0.07)

def contrastive_loss(fg_f, bg_f):
    B,D,_ = fg_f.shape
    fg_obj = fg_f.mean(dim=2); bg_obj = bg_f.mean(dim=2)
    emb    = torch.cat([fg_obj, bg_obj], dim=0)
    labels = torch.cat([torch.arange(B), torch.arange(B)]).to(fg_f.device)
    try:    return ntxent(emb, labels)
    except: return torch.tensor(0.0, device=fg_f.device)

def repulsion_loss(fg_f, bg_f):
    return F.cosine_similarity(fg_f.mean(2), bg_f.mean(2), dim=-1).mean()

def spatial_loss_uniform(feats, xyz, k=10):
    B,D,N = feats.shape
    knn_idx = torch.cdist(xyz,xyz).topk(k+1,dim=-1,largest=False).indices[:,:,1:]
    ft   = feats.permute(0,2,1).contiguous()
    flat = knn_idx.reshape(B,-1)
    nb   = torch.gather(ft,1,flat.unsqueeze(-1).expand(B,N*k,D)).view(B,N,k,D)
    fi   = ft.unsqueeze(2).expand(B,N,k,D)
    return ((fi-nb)**2).sum(-1).mean()

def entropy_loss(probs):
    return -(probs * torch.log(probs + 1e-8)).sum(-1).mean()

def confidence_guided_consistency_loss(student_probs, teacher_probs, threshold=0.8):
    teacher_probs = teacher_probs.detach()
    confidence    = teacher_probs.max(dim=-1).values        # (B, N)
    mask          = (confidence > threshold).float()        # (B, N)
    diff_sq       = ((student_probs - teacher_probs)**2).sum(dim=-1)  # (B, N)
    weighted      = mask * diff_sq
    n_confident   = mask.sum().clamp(min=1.0)
    loss          = weighted.sum() / n_confident
    frac          = mask.mean().item()
    return loss, frac

print('Loss functions defined.')

In [ ]:
# ── CELL 7: EMA + config + evaluate ──────────────────────────────────────────
from sklearn.metrics import jaccard_score, f1_score

CFG = {
    'obj_class': 4, 'num_points': 1024, 'batch_size': 8,
    'n_fg': 256, 'n_bg': 256, 'n_parts': 2, 'emb_dim': 512, 'dgcnn_k': 20
}

def build_teacher(student):
    teacher = copy.deepcopy(student)
    for p in teacher.parameters(): p.requires_grad = False
    return teacher

@torch.no_grad()
def ema_update(teacher, student, alpha=0.999):
    for t_p, s_p in zip(teacher.parameters(), student.parameters()):
        t_p.data.mul_(alpha).add_(s_p.data, alpha=(1.0 - alpha))

def evaluate(model, test_loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xyz, lbl in test_loader:
            xyz = xyz.to(DEVICE); lbl = lbl.long()
            feats, fg_f, bg_f, _, _, _ = model(xyz)
            ft   = feats.permute(0,2,1)
            fg_p = fg_f.mean(2,keepdim=True).permute(0,2,1)
            bg_p = bg_f.mean(2,keepdim=True).permute(0,2,1)
            pred = ((ft-fg_p).norm(dim=-1) < (ft-bg_p).norm(dim=-1)).long()
            preds_all.append(pred.cpu().numpy().reshape(-1))
            labels_all.append(lbl.numpy().reshape(-1))
    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(labels_all)
    iou = jaccard_score(y_true, y_pred, average='binary', zero_division=0)
    f1  = f1_score(     y_true, y_pred, average='binary', zero_division=0)
    return iou, f1

print('EMA + evaluate defined.')

In [ ]:
# ── CELL 8: Inject ALL saved results from this session ────────────────────────
# Nothing trains here. All values read directly from the notebook outputs.

saved = {
    # ── historical (previous sessions) ──────────────────────────────────────
    'baseline':             {'iou': 0.3379, 'f1': 0.5052},
    'spatial_entropy':      {'iou': 0.3832, 'f1': 0.5541},
    'ema_std_best_sess':    {'iou': 0.6086, 'f1': 0.7567},  # prev session best
    'ema_alpha_best':       {'iou': 0.6286, 'f1': 0.7719},  # alpha=0.999 ablation
    # ── this session (06_04_26) ───────────────────────────────────────────────
    'ema_standard_30ep':    {'iou': 0.3222, 'f1': 0.4874},  # Cell 11
    'conf_thresh_0.6':      {'iou': 0.1243, 'f1': 0.2211},  # Cell 13
    'conf_thresh_0.7':      {'iou': 0.4002, 'f1': 0.5717},  # Cell 13
    'conf_thresh_0.8':      {'iou': 0.6791, 'f1': 0.8089},  # Cell 13 best at E03
    # conf_thresh_0.9 not available — session cut off
}

print('Saved results loaded:')
for k,v in saved.items():
    print(f'  {k:<30} IOU={v["iou"]:.4f}  F1={v["f1"]:.4f}')

# Best threshold confirmed from ablation
BEST_THRESHOLD = 0.8
print(f'\nBest threshold: {BEST_THRESHOLD}')
print('Now running full 30-epoch confidence-guided run...')

In [ ]:
# ── CELL 9: Training function ─────────────────────────────────────────────────
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

def run_conf_guided(tag='conf_guided', n_epochs=30, lr=3e-4,
                    lambda_sp=0.01, k_sp=10, lambda_ent=0.0001,
                    ema_alpha=0.999, lambda_ema=0.1,
                    ema_warmup_epochs=5, conf_threshold=0.8):

    print(f'\n{"="*62}')
    print(f'  {tag.upper()}')
    print(f'  conf_threshold={conf_threshold} | α={ema_alpha} | λ_ema={lambda_ema} | warmup={ema_warmup_epochs}ep')
    print(f'{"="*62}')

    train_loader = DataLoader(
        ShapeNet_coseg('train', CFG['num_points'], CFG['obj_class']),
        batch_size=CFG['batch_size'], shuffle=True, drop_last=True, num_workers=2)
    test_loader = DataLoader(
        ShapeNet_coseg('test', CFG['num_points'], CFG['obj_class']),
        batch_size=CFG['batch_size'], shuffle=False, num_workers=2)

    student = CoSegNet(CFG['n_fg'], CFG['n_bg'], CFG['dgcnn_k'],
                       CFG['emb_dim'], CFG['n_parts']).to(DEVICE)
    teacher = build_teacher(student)
    opt     = optim.Adam(student.parameters(), lr=lr, weight_decay=1e-4)
    sched   = StepLR(opt, step_size=15, gamma=0.5)

    best_iou, best_f1 = 0.0, 0.0
    history = {'loss':[], 'iou':[], 'f1':[], 'iou_student':[], 'conf_frac':[]}

    for epoch in range(n_epochs):
        student.train(); teacher.eval()
        ep_loss = 0.0; ep_conf = []
        use_ema = (epoch >= ema_warmup_epochs)

        for xyz, _ in train_loader:
            xyz = xyz.to(DEVICE)
            s_feats, s_fg_f, s_bg_f, _, _, s_probs = student(xyz)
            with torch.no_grad():
                _, _, _, _, _, t_probs = teacher(xyz)

            loss = (contrastive_loss(s_fg_f, s_bg_f)
                    + 0.5  * repulsion_loss(s_fg_f, s_bg_f)
                    + lambda_sp  * spatial_loss_uniform(s_feats, xyz, k_sp)
                    + lambda_ent * entropy_loss(s_probs))

            if use_ema:
                l_ema, frac = confidence_guided_consistency_loss(
                    s_probs, t_probs, threshold=conf_threshold)
                loss = loss + lambda_ema * l_ema
                ep_conf.append(frac)

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()
            ema_update(teacher, student, alpha=ema_alpha)
            ep_loss += loss.item()

        sched.step()
        iou_t, f1_t = evaluate(teacher, test_loader)
        iou_s, _    = evaluate(student, test_loader)
        avg_loss    = ep_loss / len(train_loader)
        avg_conf    = float(np.mean(ep_conf)) if ep_conf else 0.0

        history['loss'].append(avg_loss)
        history['iou'].append(iou_t)
        history['f1'].append(f1_t)
        history['iou_student'].append(iou_s)
        history['conf_frac'].append(avg_conf)

        marker = ''
        if iou_t > best_iou:
            best_iou, best_f1 = iou_t, f1_t
            torch.save(teacher.state_dict(),
                f'/content/results/checkpoints/{tag}_teacher_best.pt')
            marker = '  --> Best'

        warmup_str = ' [warmup]' if not use_ema else ''
        conf_str   = f' | conf%={avg_conf*100:.1f}' if use_ema else ''
        print(f'[{tag}] E{epoch:02d} | loss {avg_loss:.4f} | '
              f'T-IOU {iou_t:.4f} | S-IOU {iou_s:.4f} | '
              f'F1 {f1_t:.4f}{conf_str}{marker}{warmup_str}')

    print(f'\nDONE [{tag}]  Best Teacher IOU={best_iou:.4f}  F1={best_f1:.4f}')
    return history, best_iou, best_f1

print('Training function defined.')

In [ ]:
# ── CELL 10: Full 30-epoch confidence-guided run ──────────────────────────────
# This is the run that disconnected at E03 last time.
# Threshold=0.8 was confirmed best from the ablation.

h_conf, iou_conf, f1_conf = run_conf_guided(
    tag='conf_guided_thresh0p8',
    n_epochs=30,
    ema_alpha=0.999,
    lambda_ema=0.1,
    ema_warmup_epochs=5,
    conf_threshold=0.8
)

print(f'\nConfidence-guided (thresh=0.8): IOU={iou_conf:.4f}  F1={f1_conf:.4f}')

In [ ]:
# ── CELL 11: All plots ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confidence-Guided EMA (thresh=0.8) — 30 Epochs',
             fontsize=13, fontweight='bold')

epochs = range(len(h_conf['iou']))

# Teacher IOU
axes[0].plot(epochs, h_conf['iou'], color='darkorange', lw=2, label='Teacher IOU')
axes[0].plot(epochs, h_conf['iou_student'], color='steelblue', lw=1.5,
             ls='--', alpha=0.7, label='Student IOU')
axes[0].axvline(x=5, color='gray', ls=':', alpha=0.5, label='Warmup end')
axes[0].axhline(y=saved['ema_std_best_sess']['iou'], color='red', ls=':',
                alpha=0.6, label=f'Prev best EMA (0.6086)')
axes[0].set_title('IOU over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# F1
axes[1].plot(epochs, h_conf['f1'], color='darkorange', lw=2)
axes[1].axvline(x=5, color='gray', ls=':', alpha=0.5)
axes[1].axhline(y=saved['ema_std_best_sess']['f1'], color='red', ls=':',
                alpha=0.6, label='Prev best EMA F1')
axes[1].set_title('Teacher F1 over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Confident point fraction
axes[2].plot(epochs, h_conf['conf_frac'], color='green', lw=2)
axes[2].set_title('Fraction of Confident Points (thresh=0.8)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Fraction')
axes[2].set_ylim(0, 1.05); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results/conf_guided_full_run.png', dpi=150, bbox_inches='tight')
plt.show()

# Stability analysis
half = len(h_conf['iou']) // 2
var_early = np.var(h_conf['iou'][:half])
var_late  = np.var(h_conf['iou'][half:])
mean_early = np.mean(h_conf['iou'][:half])
mean_late  = np.mean(h_conf['iou'][half:])
print(f'\nStability (lower variance = more stable):')
print(f'  Early epochs mean={mean_early:.4f}  variance={var_early:.6f}')
print(f'  Late  epochs mean={mean_late:.4f}  variance={var_late:.6f}')

In [ ]:
# ── CELL 12: Complete final summary ───────────────────────────────────────────
print('\n' + '='*65)
print('  COMPLETE RESULTS — ALL METHODS')
print('  ShapeNet Part (Chair) | DGCNN + PointSamplers | 1024 pts')
print('='*65)

all_results = [
    ('Baseline (contrastive only)',         0.3379, 0.5052),
    ('Spatial + Entropy',                   0.3832, 0.5541),
    ('EMA Standard KL (prev session)',       0.6086, 0.7567),
    ('EMA Standard KL (this session)',       0.3222, 0.4874),
    ('Conf-guided thresh=0.6 (15 ep)',       0.1243, 0.2211),
    ('Conf-guided thresh=0.7 (15 ep)',       0.4002, 0.5717),
    ('Conf-guided thresh=0.8 (15 ep)',       0.6791, 0.8089),
    (f'Conf-guided thresh=0.8 (30 ep)',      iou_conf, f1_conf),
]

print(f'  {"Method":<45} {"IOU":>8} {"F1":>8}')
print('  ' + '-'*63)
for name, iou, f1 in all_results:
    marker = '  <- BEST' if iou == max(r[1] for r in all_results) else ''
    print(f'  {name:<45} {iou:>8.4f} {f1:>8.4f}{marker}')
print('='*65)

print(f'\nThreshold ablation summary (15 epochs, α=0.999):')
print(f'  thresh=0.6 → IOU=0.1243  (too strict — almost no confident points)')
print(f'  thresh=0.7 → IOU=0.4002')
print(f'  thresh=0.8 → IOU=0.6791  <- best')
print(f'  thresh=0.9 → not completed (session disconnected)')

In [ ]:
# ── CELL 13: Save everything to Drive ─────────────────────────────────────────
import shutil
dest = '/content/drive/MyDrive/BTP_conf_guided_EMA_final'
os.makedirs(dest, exist_ok=True)
shutil.copytree('/content/results', dest, dirs_exist_ok=True)
print('Saved to Drive:', dest)
for f in sorted(os.listdir('/content/results')):
    if f.endswith('.png'): print(' ', f)